# Build Uzbek Word Frequency Dictionary
Extracts words from Uzbek Wikipedia plain text and produces `word_freq.json`.

In [ ]:
import os
import re
import sys
import json
from collections import Counter

# Make the package importable from the notebook
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from uzbek_text_tools.transliterator import transliterate

## Step 1 — Extract words from the Wikipedia plain-text dump

In [ ]:
# Cyrillic detector — if >10 % of alphabetic chars are Cyrillic, transliterate first
CYRILLIC_RE = re.compile(r'[\u0400-\u04FF]')
ALPHA_RE    = re.compile(r'[\w]')

def maybe_transliterate(text: str) -> str:
    alpha = ALPHA_RE.findall(text)
    if not alpha:
        return text
    cyr_ratio = len(CYRILLIC_RE.findall(text)) / len(alpha)
    return transliterate(text) if cyr_ratio > 0.1 else text


# Uzbek Latin word pattern — includes apostrophe letters ʻ ʼ and digraphs
WORD_RE = re.compile(r"[a-zA-ZʻʼoOgG']{2,}")

def extract_words(text: str) -> list:
    text = maybe_transliterate(text)
    return [w.lower() for w in WORD_RE.findall(text)]


EXTRACTED_DIR = os.path.join('..', 'extracted')

word_counts = Counter()
file_count = 0

for root, dirs, files in os.walk(EXTRACTED_DIR):
    for filename in files:
        filepath = os.path.join(root, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
        word_counts.update(extract_words(text))
        file_count += 1
        if file_count % 50 == 0:
            print(f'  processed {file_count} files, {len(word_counts):,} unique tokens so far')

print(f'\nDone. {file_count} files, {len(word_counts):,} total unique tokens.')

## Step 2 — Filter and inspect

In [ ]:
# Keep only words appearing ≥ 3 times
filtered = {w: c for w, c in word_counts.items() if c >= 3}
print(f'Unique words (freq ≥ 3): {len(filtered):,}')

# Top 50 most frequent — sanity check
top50 = sorted(filtered.items(), key=lambda x: -x[1])[:50]
print('\nTop 50 words:')
for word, count in top50:
    print(f'  {word:<20} {count:>8,}')

## Step 3 — Save to disk

In [ ]:
OUT_PATH = os.path.join('..', 'uzbek_text_tools', 'data', 'word_freq.json')

# Sort by frequency descending so the file is human-readable
sorted_dict = dict(sorted(filtered.items(), key=lambda x: -x[1]))

with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(sorted_dict, f, ensure_ascii=False, indent=2)

print(f'Saved {len(sorted_dict):,} words to {OUT_PATH}')

## Step 4 — Quality check
Inspect the top 500 words. Common Uzbek function words (`va`, `bu`, `bilan`, `uchun`, `ham`) should dominate. Any obvious garbage? Adjust `WORD_RE` above.

In [ ]:
top500 = sorted(sorted_dict.items(), key=lambda x: -x[1])[:500]
print('Top 500 words (word | count):')
for i, (word, count) in enumerate(top500):
    print(f'{i+1:>4}. {word:<25} {count:>8,}')

## Known limitations
- Wikipedia text is **formal**. Slang, SMS language, and casual speech are underrepresented.
- Some Wikipedia articles (especially older ones) may still be in Cyrillic — these are transliterated before word extraction.
- Plan v2: supplement with social-media corpora (Twitter/X, Telegram channels).